In [0]:
import fastf1
import pandas as pd

from fastf1 import get_session
from tqdm import tqdm
import numpy as np
import os
import tempfile

# -------------------------
# CACHE
# -------------------------
_cache_dir = os.path.join(tempfile.gettempdir(), 'fastf1_cache')
os.makedirs(_cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(_cache_dir)

In [0]:
# -------------------------
# CIRCUIT NAME LOOKUP
# -------------------------
CIRCUIT_NAME_BY_LOCATION = {
    "Melbourne": "Albert Park Circuit",
    "Sakhir": "Bahrain International Circuit",
    "Shanghai": "Shanghai International Circuit",
    "Baku": "Baku City Circuit",
    "Barcelona": "Circuit de Barcelona-Catalunya",
    "Monaco": "Circuit de Monaco",
    "Monte Carlo": "Circuit de Monaco",
    "Montreal": "Circuit Gilles Villeneuve",
    "Montr\u00e9al": "Circuit Gilles Villeneuve",
    "Le Castellet": "Circuit Paul Ricard",
    "Spielberg": "Red Bull Ring",
    "Silverstone": "Silverstone Circuit",
    "Hockenheim": "Hockenheimring",
    "Budapest": "Hungaroring",
    "Spa-Francorchamps": "Circuit de Spa-Francorchamps",
    "Monza": "Autodromo Nazionale di Monza",
    "Marina Bay": "Marina Bay Street Circuit",
    "Singapore": "Marina Bay Street Circuit",
    "Sochi": "Sochi Autodrom",
    "Suzuka": "Suzuka Circuit",
    "Austin": "Circuit of the Americas",
    "Mexico City": "Aut\u00f3dromo Hermanos Rodr\u00edguez",
    "S\u00e3o Paulo": "Aut\u00f3dromo Jos\u00e9 Carlos Pace",
    "Sao Paulo": "Aut\u00f3dromo Jos\u00e9 Carlos Pace",
    "Interlagos": "Aut\u00f3dromo Jos\u00e9 Carlos Pace",
    "Yas Marina": "Yas Marina Circuit",
    "Yas Island": "Yas Marina Circuit",
    "Abu Dhabi": "Yas Marina Circuit",
    "Mugello": "Autodromo Internazionale del Mugello",
    "N\u00fcrburg": "N\u00fcrburgring",
    "Nuerburg": "N\u00fcrburgring",
    "N\u00fcrburgring": "N\u00fcrburgring",
    "Portim\u00e3o": "Aut\u00f3dromo Internacional do Algarve",
    "Portimao": "Aut\u00f3dromo Internacional do Algarve",
    "Imola": "Autodromo Enzo e Dino Ferrari",
    "Istanbul": "Istanbul Park",
    "Zandvoort": "Circuit Zandvoort",
    "Jeddah": "Jeddah Corniche Circuit",
    "Losail": "Losail International Circuit",
    "Lusail": "Losail International Circuit",
    "Miami": "Miami International Autodrome",
    "Las Vegas": "Las Vegas Strip Circuit",
}


def circuit_name_for(location):
    return CIRCUIT_NAME_BY_LOCATION.get(location, location)


In [0]:
session = get_session(2023, 'Silverstone', 'R')  # Year, GP name, session type (R=Race, Q=Quali)
session.load()  # downloads data and parses everything

In [0]:
laps = session.laps  # lap-level data (driver, time, stint, tyre, pit stop)
results = session.results  # final positions, fastest lap, team
weather = session.weather_data  # temp, humidity, wind
track_stats = session.track_status

In [0]:
pd.set_option('display.max_columns', None)

In [0]:
laps.loc[laps["TrackStatus"] != "1"].head(1000)

In [0]:
results.head()

In [0]:
weather.head()

In [0]:
track_stats.head(150)

In [0]:
events = fastf1.get_event_schedule(2018)
events.head()

In [0]:
events = fastf1.get_event_schedule(2020)
events = events.loc[events["EventFormat"] == "conventional"]["Location"]
events = list(set(events.tolist()))
print(events)

In [0]:
years = [2018, 2019, 2020, 2021, 2022, 2023]
event_data = {}
for year in years:
    e = fastf1.get_event_schedule(year)
    e = e.loc[e["EventFormat"] != "testing"]["EventName"]
    races = e.tolist()
    event_data[year] = races

In [0]:
print(event_data)
for year, events in event_data.items():
    print(f"\t{year} - {len(events)} - {events}")

In [0]:
TABLE_NAME = "workspace.default.f1_master_lap_dataset"

# -------------------------
# RESUME SUPPORT
# -------------------------
already_ingested = set()
if spark.catalog.tableExists(TABLE_NAME):
    _existing = spark.table(TABLE_NAME).select("Year", "Race").distinct().toPandas()
    already_ingested = set(zip(_existing["Year"], _existing["Race"]))
    print(f"Resuming: {len(already_ingested)} races already in {TABLE_NAME} — will be skipped.")

master_rows = []

for year in tqdm(event_data, desc="Years"):
    for race in tqdm(event_data[year], desc=f"Processing {year}", leave=False):

        if (year, race) in already_ingested:
            continue

        try:
            # -------------------------
            # LOAD SESSION (RACE)
            # -------------------------
            session = fastf1.get_session(year, race, 'R')
            session.load(telemetry=False)

            laps = session.laps.copy()
            weather = session.weather_data.copy()
            drivers = session.drivers

            # -------------------------
            # CLEAN WEATHER COLUMNS
            # -------------------------
            weather = weather.rename(columns={
                'AirTemp': 'AirTemp_C',
                'TrackTemp': 'TrackTemp_C',
                'Humidity': 'Humidity_pct',
                'WindSpeed': 'WindSpeed_kmh',
                'Rainfall': 'Rainfall_mm'
            })

            # Sort for merge_asof()
            laps_sorted = laps.sort_values("Time")
            weather_sorted = weather.sort_values("Time")

            # -------------------------
            # MERGE WEATHER INTO LAPS
            # -------------------------
            laps_merged = pd.merge_asof(
                laps_sorted,
                weather_sorted,
                on="Time",
                direction="nearest"
            )

            # -------------------------
            # ENSURE TYRE-RELATED FIELDS EXIST
            # -------------------------
            tyre_fields = ['Compound', 'Stint', 'FreshTyre', 'TyreLife']
            for col in tyre_fields:
                if col not in laps_merged.columns:
                    laps_merged[col] = None

            # -------------------------
            # ADD SESSION METADATA
            # -------------------------
            laps_merged['Year'] = year
            laps_merged['Race'] = race
            _location = session.event['Location']
            laps_merged['Circuit'] = circuit_name_for(_location)
            laps_merged['Location'] = _location
            if _location not in CIRCUIT_NAME_BY_LOCATION:
                print(f"\u26a0\ufe0f No circuit-name mapping for Location={_location!r} ({year} {race}) -- using raw Location as Circuit.")
            laps_merged['Country'] = session.event['Country']
            laps_merged['SessionDate'] = session.event['EventDate']

            # -------------------------
            # ADD DRIVER METADATA
            # -------------------------
            driver_info = {}
            for drv in drivers:
                info = session.get_driver(drv)
                driver_info[info['Abbreviation']] = {
                    'DriverNumber': info['DriverNumber'],
                    'BroadcastName': info['BroadcastName'],
                    'TeamColor': info['TeamColor'],
                    'TeamName': info['TeamName']
                }

            laps_merged['DriverNumber'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('DriverNumber')
            )
            laps_merged['TeamName'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('TeamName')
            )
            laps_merged['TeamColor'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('TeamColor')
            )
            laps_merged['BroadcastName'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('BroadcastName')
            )

            # -------------------------
            # TRACK STATUS FLAGS
            # -------------------------
            # 1 = track clear, 2 = yellow, 4 = SC, 5 = VSC, 7 = red flag
            laps_merged['TrackStatus'] = laps_merged['TrackStatus'].fillna("1")

            # -------------------------
            # SECTOR TIMES (important for degradation analysis)
            # -------------------------
            laps_merged['Sector1Time'] = laps_merged['Sector1Time'].fillna(pd.Timedelta(seconds=0))
            laps_merged['Sector2Time'] = laps_merged['Sector2Time'].fillna(pd.Timedelta(seconds=0))
            laps_merged['Sector3Time'] = laps_merged['Sector3Time'].fillna(pd.Timedelta(seconds=0))

            # -------------------------
            # GAP TO CAR AHEAD
            # -------------------------
            laps_merged['Position'] = pd.to_numeric(laps_merged['Position'], errors='coerce')

            _by_pos = laps_merged.sort_values(['LapNumber', 'Position'])
            gap_to_ahead = _by_pos.groupby('LapNumber')['Time'].diff().dt.total_seconds()
            laps_merged['GapToAhead'] = gap_to_ahead.reindex(laps_merged.index)
            laps_merged.loc[laps_merged['Position'].isna(), 'GapToAhead'] = np.nan

            # -------------------------
            # DELTA TO LEADER
            # -------------------------
            leader_time_by_lap = (
                laps_merged.loc[laps_merged['Position'] == 1]
                .drop_duplicates('LapNumber')
                .set_index('LapNumber')['Time']
            )
            laps_merged['DeltaToLeader'] = (
                laps_merged['Time'] - laps_merged['LapNumber'].map(leader_time_by_lap)
            ).dt.total_seconds()
            laps_merged.loc[laps_merged['Position'].isna(), 'DeltaToLeader'] = np.nan

            # -------------------------
            # DELTA TO FIELD AVERAGE
            # -------------------------
            avg_laptimes = laps_merged.groupby("LapNumber")["LapTime"].transform("mean")
            laps_merged["DeltaToAverage"] = (laps_merged["LapTime"] - avg_laptimes)

            # -------------------------
            # PIT STOP PROCESSING
            # -------------------------
            _pit_in_raw = pd.to_timedelta(laps_merged["PitInTime"], errors="coerce")
            _pit_out_raw = pd.to_timedelta(laps_merged["PitOutTime"], errors="coerce")

            _sorted_idx = laps_merged.sort_values(["Driver", "LapNumber"]).index
            _prev_pit_in = (
                _pit_in_raw.reindex(_sorted_idx)
                .groupby(laps_merged["Driver"].reindex(_sorted_idx))
                .shift(1)
                .reindex(laps_merged.index)
            )

            laps_merged["HasPit"] = _pit_out_raw.notna() & _prev_pit_in.notna()
            laps_merged["PitDuration"] = np.where(
                laps_merged["HasPit"],
                (_pit_out_raw - _prev_pit_in).dt.total_seconds(),
                0
            )

            # Clean negative durations (bad data in old races)
            laps_merged.loc[laps_merged["PitDuration"] < 0, "PitDuration"] = 0

            laps_merged["PitInTime"] = _pit_in_raw
            laps_merged["PitOutTime"] = _pit_out_raw

            # -------------------------
            # RACE LAP NUMBER
            # -------------------------
            laps_merged['RaceLap'] = laps_merged['LapNumber']

            # -------------------------
            # QUALIFYING RESULTS
            # -------------------------
            try:
                # telemetry=False: only quali.results is used below, never
                # laps/telemetry from the qualifying session.
                quali = fastf1.get_session(year, race, 'Q')
                quali.load(telemetry=False)

                quali_results = quali.results[['DriverNumber', 'Position', 'Q1', 'Q2', 'Q3']]
                quali_results = quali_results.rename(columns={
                    'Position': 'QualiPosition',
                    'Q1': 'Q1Time',
                    'Q2': 'Q2Time',
                    'Q3': 'Q3Time'
                })

                laps_merged = laps_merged.merge(
                    quali_results,
                    on='DriverNumber',
                    how='left'
                )

            except Exception as e:
                print(f"⚠️ Qualifying not available for {race} {year}: {e}")
                laps_merged[['QualiPosition', 'Q1Time', 'Q2Time', 'Q3Time']] = None

            # -------------------------
            # FINAL RACE RESULTS
            # -------------------------
            results = session.results[['DriverNumber', 'Position', 'Status', 'Points', 'Time']]
            results = results.rename(columns={
                'Position': 'FinalPosition',
                'Time': 'FinalRaceTime'
            })

            laps_merged = laps_merged.merge(
                results,
                on='DriverNumber',
                how='left'
            )

            # -------------------------
            # NORMALIZE DURATION COLUMNS TO PLAIN SECONDS (PER RACE)
            # -------------------------
            _timedelta_cols = laps_merged.select_dtypes(include=["timedelta64[ns]"]).columns
            for _col in _timedelta_cols:
                laps_merged[_col] = laps_merged[_col].dt.total_seconds()

            # -------------------------
            # WRITE THIS RACE TO THE UNITY CATALOG TABLE IMMEDIATELY
            # -------------------------
            spark.createDataFrame(laps_merged).write.mode("append").saveAsTable(TABLE_NAME)

            # Race control messages (track limits / penalties, for Plan G)
            rc = session.race_control_messages.copy()
            rc['Year'] = year
            rc['Race'] = race
            rc['Circuit'] = circuit_name_for(_location)
            _rc_timedelta_cols = rc.select_dtypes(include=["timedelta64[ns]"]).columns
            for _col in _rc_timedelta_cols:
                rc[_col] = rc[_col].dt.total_seconds()
            if not rc.empty:
                spark.createDataFrame(rc).write.mode("append").saveAsTable("workspace.default.f1_race_control_messages")

            # -------------------------
            # APPEND TO MASTER (same-session debug artifact only)
            # -------------------------
            master_rows.append(laps_merged)
            print("-"*100)
            print(f"Success for {year} {race}!!!!!")
            print("-"*100)

        except fastf1.exceptions.RateLimitExceededError:
            print(f"⏱️ Rate limit hit at {race} {year} — stopping this run early. Re-run later to resume.")
            raise SystemExit("FastF1 rate limit reached — stopping run; re-run later to resume.")

        except Exception as e:
            print(f"⚠️ FAILED FOR {race} {year} → {e}")
            continue


# -------------------------
# SAVE (same-session local artifact only — each race was already written
# to the Unity Catalog table above; this is just for in-session debugging)
# -------------------------
if master_rows:
    master_df = pd.concat(master_rows, ignore_index=True)
    master_df.to_parquet("../data/f1_master_lap_dataset.parquet", index=False)
    print("🎉 LOCAL PARQUET SAVED (session-local only, this run's new races) → ../data/f1_master_lap_dataset.parquet")
    print(f"New rows this run: {len(master_df)}")
else:
    print("No new races fetched this run (all already ingested, or all failed).")

In [0]:
print(f"Total rows now in {TABLE_NAME}: {spark.table(TABLE_NAME).count()}")